# 实验6.2 基于 DVPP 与 AIPP 的 YOLO 目标检测与 DeepSort 多目标追踪

> 昇腾 910B3 NPU · YOLOv8 目标检测 · DVPP 硬件解码 + AIPP 预处理加速 + DeepSort 多目标追踪

本实验在 **gitcode CANNLab 云平台** 上运行，采用 **ASCEND 1*NPU 910B3** 硬件配置，围绕 **YOLOv8 目标检测模型** 的部署与 **DeepSort 多目标追踪** 展开。实验重点实践昇腾独有的 **DVPP** 硬件视频解码与 **AIPP** 硬件预处理加速能力，并结合 DeepSort 实现视频流中的多目标连续追踪。

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">项目</th>
<th style="text-align: left;">内容</th>
</tr>
<tr>
<td style="text-align: left;"><strong>云平台</strong></td>
<td style="text-align: left;">gitcode CANNLab</td>
</tr>
<tr>
<td style="text-align: left;"><strong>硬件配置</strong></td>
<td style="text-align: left;">ASCEND, 1*NPU 910B3, 16vCPUs, 32GiB</td>
</tr>
<tr>
<td style="text-align: left;"><strong>NPU</strong></td>
<td style="text-align: left;">昇腾 910B3</td>
</tr>
<tr>
<td style="text-align: left;"><strong>CPU</strong></td>
<td style="text-align: left;">16 vCPUs</td>
</tr>
<tr>
<td style="text-align: left;"><strong>内存</strong></td>
<td style="text-align: left;">32 GiB</td>
</tr>
<tr>
<td style="text-align: left;"><strong>软件环境</strong></td>
<td style="text-align: left;">CANN Toolkit · ATC · AscendCL · OpenCV</td>
</tr>
<tr>
<td style="text-align: left;"><strong>检测模型</strong></td>
<td style="text-align: left;">YOLOv8n（目标检测）</td>
</tr>
<tr>
<td style="text-align: left;"><strong>追踪算法</strong></td>
<td style="text-align: left;">DeepSort（卡尔曼滤波 + 匈牙利匹配）</td>
</tr>
</table>

---

### 视频素材准备

`images/dog.mp4` 为本实验所用测试视频，体积较大，**不再随仓库分发**。下方单元格会在文件缺失时自动从在线地址下载到 `images/dog.mp4`；若已手动放置则跳过。


In [ ]:
import os, urllib.request

def _ensure_dog_video(path='images/dog.mp4', url='https://www.qmpan.com/f/6pLXiD/dog.mp4'):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    if os.path.exists(path):
        return
    print(f'[INFO] {path} 不存在，正在从在线地址下载...')
    urllib.request.urlretrieve(url, path)
    print(f'[OK] 已下载到 {path}')

_ensure_dog_video()


## 1. 实验概述与目标

### 1.1 实验背景

在端侧设备（开发板、边缘盒子）上部署视觉模型，面临三大挑战：

1. **算力有限**：端侧 NPU 算力远小于服务器，必须极致优化
2. **预处理开销大**：JPEG 解码、缩放、归一化等预处理在 CPU 上耗时占比高
3. **带宽受限**：Host→Device 数据搬运是瓶颈

昇腾通过 **DVPP + AIPP** 两大硬件加速能力解决这些问题：

- **DVPP**：把 JPEG 解码和图像缩放从 CPU 移到 NPU 专用硬件
- **AIPP**：把归一化和通道转换从 CPU 移到 NPU 专用硬件，并将输入数据从 float32 降为 uint8

### 1.2 实验目标

- **知识目标**：理解 DVPP 硬件视频解码加速原理；理解 AIPP 预处理卸载机制；掌握 YOLO 模型端侧部署全流程
- **能力目标**：能够完成 YOLOv8 模型的 ONNX 导出、ATC 转换与 OM 推理；能够对比 DVPP/OpenCV 预处理与 AIPP/无 AIPP 推理路径的性能差异
- **素养目标**：形成"实践→总结→贡献"的闭环意识，积极向 CANN 社区反馈端侧部署经验

## 2. 实验环境：gitcode CANNLab 云平台

本实验在 **gitcode CANNLab 云平台** 上运行，该平台提供昇腾 NPU 云算力环境，无需本地部署即可进行 CANN 相关实验。

<table style="text-align: left; margin-left: 0;">
<tr><th style="text-align: left;">规格</th><th style="text-align: left;">参数</th></tr>
<tr><td style="text-align: left;">云平台</td><td style="text-align: left;">gitcode CANNLab</td></tr>
<tr><td style="text-align: left;">硬件配置</td><td style="text-align: left;">ASCEND, 1*NPU 910B3, 16vCPUs, 32GiB</td></tr>
<tr><td style="text-align: left;">NPU</td><td style="text-align: left;">昇腾 910B3（单卡）</td></tr>
<tr><td style="text-align: left;">CPU</td><td style="text-align: left;">16 vCPUs（虚拟 CPU 核心）</td></tr>
<tr><td style="text-align: left;">内存</td><td style="text-align: left;">32 GiB</td></tr>
<tr><td style="text-align: left;">操作系统</td><td style="text-align: left;">Linux（x86_64）</td></tr>
<tr><td style="text-align: left;">CANN 版本</td><td style="text-align: left;">CANN Toolkit (Ascend910B3)</td></tr>
<tr><td style="text-align: left;">Python</td><td style="text-align: left;">3.11.x</td></tr>
</table>

上表列出了实验所用的云平台关键规格。**NPU** 是昇腾 910B3 单卡，AI 算力远大于端侧 310B4，适合运行 YOLOv8n 等轻量级模型推理及 DeepSort 追踪算法。**CPU** 配置 16 vCPUs，可并行执行 OpenCV 预处理、后处理和追踪逻辑。**内存** 32 GiB 充足，可缓存完整视频帧序列用于 DeepSort 追踪。与端侧开发板不同，云平台采用 Host-Device 分离架构，需通过 `acl.rt.memcpy` 进行 H2D/D2H 数据搬运，但 DVPP 和 AIPP 的核心概念与端侧完全一致。

> **注意**：本 Notebook 直接运行在 gitcode CANNLab 云平台的昇腾 910B3 环境上，实验代码中 `soc_version` 使用 `Ascend910B3`。

## 3. DVPP 硬件视频解码详解

### 3.1 DVPP 是什么

DVPP（Digital Vision Pre-Processing）是昇腾 NPU 内部的专用视频处理引擎，独立于 AI Core，专门处理图像/视频的编解码与几何变换。

<img src="../../images/dvpp_flow.png" alt="DVPP流水线" style="display: block; margin-left: 0;" />

### 3.2 DVPP 支持的操作

<table style="text-align: left; margin-left: 0;">
<tr><th style="text-align: left;">操作</th><th style="text-align: left;">AscendCL API</th><th style="text-align: left;">说明</th></tr>
<tr><td style="text-align: left;">JPEG 解码</td><td style="text-align: left;">acl.media.dvpp_jpeg_decode_async</td><td style="text-align: left;">JPEG → YUV420SP</td></tr>
<tr><td style="text-align: left;">VPC 缩放</td><td style="text-align: left;">acl.media.dvpp_vpc_resize_async</td><td style="text-align: left;">硬件缩放 + 裁剪</td></tr>
<tr><td style="text-align: left;">JPEG 编码</td><td style="text-align: left;">acl.media.dvpp_jpeg_encode_async</td><td style="text-align: left;">YUV420SP → JPEG</td></tr>
<tr><td style="text-align: left;">内存分配</td><td style="text-align: left;">acl.media.dvpp_malloc</td><td style="text-align: left;">DVPP 专用设备内存</td></tr>
</table>

上表列出了 DVPP 支持的四类操作。**JPEG 解码**将压缩的 JPEG 数据解码为 YUV420SP（NV12）格式，输出直接存放在 Device 内存。**VPC 缩放**不仅支持缩放，还支持裁剪、填充等几何变换。**JPEG 编码**将 YUV420SP 编码为 JPEG，常用于结果保存。**内存分配**使用 `dvpp_malloc` 而非普通 `rt.malloc`，因为 DVPP 要求内存地址按特定方式对齐。所有 DVPP 操作都是异步的，需通过 `synchronize_stream` 等待完成。

### 3.3 DVPP 对齐规则

DVPP 对输入输出尺寸有**对齐要求**，这是初学者最常踩的坑：

- 宽度对齐：128 字节对齐（`align_up(w, 128)`）
- 高度对齐：16 字节对齐（`align_up(h, 16)`）
- 输出格式：YUV420SP（NV12），半平面存储

> 对齐函数：`align_up(size, align) = (size + align - 1) // align * align`

## 4. AIPP 预处理加速详解

### 4.1 AIPP 卸载机制

<img src="../../images/aipp_compare.png" alt="AIPP对比" style="display: block; margin-left: 0;" />

AIPP 接管了预处理四步中的两步：

<table style="text-align: left; margin-left: 0;">
<tr><th style="text-align: left;">预处理步骤</th><th style="text-align: left;">无 AIPP（CPU）</th><th style="text-align: left;">有 AIPP（NPU 硬件）</th></tr>
<tr><td style="text-align: left;">BGR→RGB</td><td style="text-align: left;">Python/OpenCV</td><td style="text-align: left;">Python/OpenCV</td></tr>
<tr><td style="text-align: left;">letterbox 缩放</td><td style="text-align: left;">Python/OpenCV</td><td style="text-align: left;">Python/OpenCV</td></tr>
<tr><td style="text-align: left;">归一化 /255</td><td style="text-align: left;">Python/CPU</td><td style="text-align: left;">AIPP 硬件（NPU）</td></tr>
<tr><td style="text-align: left;">HWC→CHW 转置</td><td style="text-align: left;">Python/CPU</td><td style="text-align: left;">AIPP 硬件（NPU）</td></tr>
<tr><td style="text-align: left;">输入数据类型</td><td style="text-align: left;">float32（4 字节）</td><td style="text-align: left;">uint8（1 字节，1/4 带宽）</td></tr>
</table>

上表详细对比了有无 AIPP 时预处理各步骤的执行位置。**BGR→RGB 色彩转换**和 **letterbox 缩放**在两种路径下都由 Python/OpenCV 完成，因为这两个操作涉及非标准几何变换（letterbox 需要保持宽高比并填充灰色边框），AIPP 硬件不擅长处理。**归一化 /255** 和 **HWC→CHW 通道转置**是每次推理都必须做的标准化操作，AIPP 将它们从 CPU 移到 NPU 硬件，这是 AIPP 的核心价值。最关键的是**输入数据类型**的变化：无 AIPP 时需传入 float32（每像素 4 字节），有 AIPP 时只需传入 uint8（每像素 1 字节），Host→Device 传输量降为原来的 1/4，在端侧带宽受限的场景下收益显著。

### 4.2 AIPP 配置文件

实验使用的 `aipp_face.cfg` 配置：

```protobuf
aipp_op {
  aipp_mode: static          # 静态 AIPP，转换时固化进 OM
  related_input_rank: 0      # 作用于第 0 号输入
  input_format: RGB888_U8    # 输入为 RGB uint8
  src_image_size_w: 640
  src_image_size_h: 640
  crop: false                # 不裁剪
  csc_switch: false          # 关闭色域转换（BGR→RGB 已由 Python 完成）
  # 归一化: y = (x - min_chn) * var_reci_chn，即 y = x / 255
  min_chn_0: 0.0
  var_reci_chn_0: 0.00392157   # = 1/255
}
```

> **关键**：`csc_switch: false` 因为 BGR→RGB 已由 Python 完成，AIPP 不重复做。"同一件事做两遍"是 AIPP 配置最典型的错误。

## 5. YOLOv8 模型部署全流程

### 5.1 部署链路

```text
YOLOv8n-face (.pt) → ONNX 导出 → ATC 转换（含 AIPP）→ OM 模型 → AscendCL NPU 推理
                                                       ↓
                                         DVPP 硬件解码 → AIPP 硬件预处理 → AI Core 推理
```

### 5.2 ATC 转换命令

实验中的 `export_om.sh` 脚本：

```bash
atc \
    --model=face_yolov8n.onnx \
    --framework=5 \
    --output=face_yolov8n \
    --soc_version=Ascend310B4 \
    --input_format=NCHW \
    --input_shape="images:1,3,640,640" \
    --insert_op_conf=aipp_face.cfg \
    --output_type=FP32
```

### 5.3 三条推理路径

实验对比了三条预处理路径：

<table style="text-align: left; margin-left: 0;">
<tr><th style="text-align: left;">路径</th><th style="text-align: left;">预处理方式</th><th style="text-align: left;">说明</th></tr>
<tr><td style="text-align: left;">Path A</td><td style="text-align: left;">OpenCV 公平版</td><td style="text-align: left;">JPEG→YUV→逐平面缩放→RGB（模拟DVPP流程）</td></tr>
<tr><td style="text-align: left;">Path B</td><td style="text-align: left;">OpenCV 最优版</td><td style="text-align: left;">JPEG→BGR→resize→RGB（直接resize）</td></tr>
<tr><td style="text-align: left;">Path C</td><td style="text-align: left;">DVPP</td><td style="text-align: left;">JPEG→硬件解码→VPC缩放→YUV→RGB</td></tr>
</table>

上表对比了三条预处理路径。**Path A（OpenCV 公平版）**故意模拟 DVPP 的处理流程：先解码为 YUV，再逐平面缩放，最后转 RGB。这条路径最慢，因为它做了不必要的 YUV 中间转换，目的是与 DVPP 做公平对比（两者处理步骤相同，只是 CPU vs 硬件）。**Path B（OpenCV 最优版）**是 OpenCV 的最佳实践：直接解码为 BGR，用 `resize` 缩放，再转 RGB。这条路径比 Path A 快，因为避免了 YUV 中间格式。**Path C（DVPP）**用 NPU 硬件完成解码和缩放，是最快的路径。实验结果通常为 Path C < Path B < Path A，DVPP 相比 OpenCV 最优版加速 2~5 倍，相比公平版加速更大。

> **为什么需要 Path A**：直接对比 Path B 和 Path C 不够公平，因为 Path B 用了 OpenCV 的优化技巧（跳过 YUV 中间格式），而 DVPP 必须经过 YUV。Path A 让 CPU 也走 YUV 路径，这样 Path A vs Path C 纯粹是 CPU vs 硬件的差异，更能体现 DVPP 的硬件加速价值。

---

## 6. 动手实践：在 NPU 上体验 DVPP 与 AIPP

> 以下代码在昇腾 910B3 云沙箱上运行，演示与香橙派实验相同的核心概念。

### 6.1 检查环境与 NPU 信息

**测试程序说明**：首先自动安装缺失的 Python 依赖（`onnx`、`opencv-python-headless`），并预加载 `libgomp` 修复 PyTorch TLS 问题。然后调用 `npu-smi info` 查询 NPU 状态，检查 `atc` 命令是否可用，以及 `acl`（AscendCL）模块是否安装。

**预期结果**：显示 NPU 设备信息（芯片型号、利用率、显存等），ATC 路径指向 CANN 安装目录，AscendCL 模块可用。如果 `atc` 未找到，需在终端执行 `source /usr/local/Ascend/ascend-toolkit/set_env.sh` 加载 CANN 环境变量。

In [ ]:
import os, sys, time, subprocess
import numpy as np

# === 环境准备：自动安装缺失依赖 ===
print('=' * 55)
print('  环境依赖检查与自动安装')
print('=' * 55)

def _ensure_pkg(pkg, import_name=None):
    import_name = import_name or pkg
    try:
        __import__(import_name)
        print(f'  [✓] {pkg} 已安装')
        return True
    except ImportError:
        print(f'  [!] {pkg} 未安装，正在安装...')
        try:
            subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])
            print(f'  [✓] {pkg} 安装完成')
            return True
        except Exception as e:
            print(f'  [✗] {pkg} 安装失败: {e}')
            return False

_ensure_pkg('onnx')
_ensure_pkg('opencv-python-headless', 'cv2')
_ensure_pkg('matplotlib')

# 修复 PyTorch TLS 加载问题（libgomp 无法分配静态 TLS 块）
try:
    import ctypes, glob
    for p in glob.glob('/opt/**/torch.libs/libgomp*.so*', recursive=True):
        try:
            ctypes.CDLL(p, mode=ctypes.RTLD_GLOBAL)
        except Exception:
            pass
except Exception:
    pass

print()
print('=' * 55)
print('  gitcode CANNLab 云平台环境信息')
print('=' * 55)

# 打印硬件配置信息
print('  云平台: gitcode CANNLab')
print('  硬件配置: ASCEND, 1*NPU 910B3, 16vCPUs, 32GiB')
import multiprocessing
print(f'  CPU 核心数: {multiprocessing.cpu_count()} vCPUs')
try:
    with open('/proc/meminfo') as f:
        for line in f:
            if line.startswith('MemTotal:'):
                mem_kb = int(line.split()[1])
                print(f'  内存总量: {mem_kb / 1024**2:.1f} GiB')
                break
except Exception:
    print('  内存总量: 32 GiB (配置)')

print()
print('=' * 55)
print('  昇腾 NPU 环境检查')
print('=' * 55)

# 检查 npu-smi
try:
    r = subprocess.run(['npu-smi', 'info'], capture_output=True, text=True, timeout=10)
    for line in r.stdout.split('\n')[:10]:
        print(line)
except Exception as e:
    print(f'npu-smi: {e}')

# 检查 ATC
r = subprocess.run(['which', 'atc'], capture_output=True, text=True)
print(f'\nATC 路径: {r.stdout.strip() if r.returncode == 0 else "未找到（请 source CANN 环境）"}')

# 检查 acl 模块
try:
    import acl
    print(f'AscendCL (acl) 模块: 可用')
except ImportError:
    print('AscendCL (acl) 模块: 未安装')

### 6.2 案例一：图片目标检测（dog1.jpg + dog2.jpg + cat1.jpg + cat2.jpg）

使用 AscendCL 的 DVPP 接口对 **四张真实图片** `../../images/dog1.jpg`、`../../images/dog2.jpg`（狗）和 `../../images/cat1.jpg`、`../../images/cat2.jpg`（猫）进行 JPEG 硬件解码与缩放。由于 DVPP 仅支持 JPEG 格式解码，需先将 PNG 读取并转换为 JPEG。图片统一调整到 **640×480**（宽度 128 字节对齐、高度 16 字节对齐，满足 DVPP 对齐要求）。

**测试程序说明**：第一个单元格从 `images/` 目录读取四张真实 PNG 图片，用 OpenCV 调整到 640×480 并保存为 JPEG（DVPP 要求 JPEG 输入），同时对四张图片模拟 YOLO 目标检测可视化。第二个单元格调用 AscendCL DVPP API 依次对四张图片完成硬件解码和缩放：① 初始化 ACL 并创建 DVPP 通道；② 读取 JPEG 数据并拷贝到 Device；③ 调用 `dvpp_jpeg_decode_async` 硬件解码为 YUV420SP；④ 调用 `dvpp_vpc_resize_async` 硬件缩放从 640×480 到 224×224；⑤ 释放所有资源。

**预期结果**：四张 PNG 图片读取并转换尺寸成功（640×480 JPEG）。DVPP 通道创建成功，每张图片 JPEG 解码约 0.3~1 ms，VPC 缩放约 0.2~0.5 ms，单张总耗时约 0.5~1.5 ms。如果 `acl` 模块未安装，会显示提示信息。

**为什么有这样的结果**：DVPP 是 NPU 内部的专用硬件电路（非 AI Core），解码和缩放由固化的硬件逻辑执行，不占用 AI Core 算力。DVPP 仅支持 JPEG 解码，因此需先将 PNG 转为 JPEG。解码输出为 YUV420SP（NV12）格式而非 RGB，因为 JPEG 内部就是以 YUV 色彩空间压缩的，直接输出 YUV 避免了不必要的色彩转换。宽高需按 128/16 对齐是因为 DVPP 硬件的 DMA 引擎按对齐块传输数据。

In [ ]:
try:
    import cv2
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'opencv-python-headless', '-q'])
    import cv2

# === 案例一：读取四张真实 PNG 图片，转换为 JPEG 供 DVPP 处理 ===
# DVPP 仅支持 JPEG 解码，因此需先将 PNG 转为 JPEG
# 目标尺寸 640x480：宽度 128 字节对齐、高度 16 字节对齐，满足 DVPP 要求
os.makedirs('output', exist_ok=True)
DVPP_W, DVPP_H = 640, 480  # DVPP 对齐友好尺寸

image_samples = [
    {'png': '../../images/dog1.jpg', 'jpg': 'output/dog1.jpg', 'label': 'Dog1'},
    {'png': '../../images/dog2.jpg', 'jpg': 'output/dog2.jpg', 'label': 'Dog2'},
    {'png': '../../images/cat1.jpg', 'jpg': 'output/cat1.jpg', 'label': 'Cat1'},
    {'png': '../../images/cat2.jpg', 'jpg': 'output/cat2.jpg', 'label': 'Cat2'},
]

test_images = []  # 保存处理后的图片信息，供后续 DVPP 单元格使用
for s in image_samples:
    src_img = cv2.imread(s['png'])
    if src_img is None:
        raise FileNotFoundError(f"无法读取 {s['png']}")
    resized = cv2.resize(src_img, (DVPP_W, DVPP_H))
    cv2.imwrite(s['jpg'], resized)
    test_images.append({
        'png': s['png'], 'jpg': s['jpg'], 'label': s['label'],
        'img': resized, 'size': os.path.getsize(s['jpg']),
    })
    print(f"  [✓] {s['png']} → {s['jpg']} ({os.path.getsize(s['jpg'])} bytes, {DVPP_W}x{DVPP_H})")

test_jpg = test_images[0]['jpg']  # 兼容后续单元格变量
print(f'\n共准备 {len(test_images)} 张测试图片，尺寸 {DVPP_W}x{DVPP_H} (来源: dog1+dog2+cat1+cat2)')

# === 模拟 YOLO 目标检测结果可视化（两张图片） ===
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

def mock_detect(img, label, conf=0.90):
    """在图像中心区域画一个模拟检测框"""
    result = img.copy()
    h, w = img.shape[:2]
    cx, cy = w // 2, h // 2
    box_w, box_h = int(w * 0.4), int(h * 0.5)
    x1, y1 = cx - box_w // 2, cy - box_h // 2
    x2, y2 = cx + box_w // 2, cy + box_h // 2
    cv2.rectangle(result, (x1, y1), (x2, y2), (0, 255, 0), 2)
    text = f'{label} {conf:.2f}'
    cv2.rectangle(result, (x1, y1 - 20), (x1 + 90, y1), (0, 255, 0), -1)
    cv2.putText(result, text, (x1 + 2, y1 - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 1)
    return result

fig, axes = plt.subplots(4, 2, figsize=(14, 20))
for i, t in enumerate(test_images):
    result_img = mock_detect(t['img'], t['label'])
    axes[i, 0].imshow(cv2.cvtColor(t['img'], cv2.COLOR_BGR2RGB))
    axes[i, 0].set_title(f"Original ({t['label']}, {os.path.basename(t['png'])}, {DVPP_W}x{DVPP_H})"); axes[i, 0].axis('off')
    axes[i, 1].imshow(cv2.cvtColor(result_img, cv2.COLOR_BGR2RGB))
    axes[i, 1].set_title(f'Mock YOLO Detection ({t["label"]})'); axes[i, 1].axis('off')
plt.suptitle('Case 1: DVPP + AIPP + YOLO Detection (dog1+dog2+cat1+cat2)', fontsize=14)
plt.tight_layout()
plt.savefig('output/yolo_detection_demo.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'检测可视化已保存: output/yolo_detection_demo.png')

In [ ]:
# DVPP 硬件解码演示（依次处理 dog1.jpg, dog2.jpg, cat1.jpg, cat2.jpg 四张图片）
try:
    import acl

    ACL_SUCCESS = 0
    ACL_MEMCPY_HOST_TO_DEVICE = 1
    ACL_MEMCPY_DEVICE_TO_HOST = 2
    PIXEL_FORMAT_YUV_SEMIPLANAR_420 = 1

    def align_up(size, align):
        return (size + align - 1) // align * align

    # 初始化 ACL
    ret = acl.init()
    assert ret == 0, f'acl.init failed: {ret}'
    ret = acl.rt.set_device(0)
    assert ret == 0
    stream, ret = acl.rt.create_stream()
    assert ret == 0

    # 创建 DVPP 通道
    dvpp_desc = acl.media.dvpp_create_channel_desc()
    ret = acl.media.dvpp_create_channel(dvpp_desc)
    assert ret == 0, f'dvpp_create_channel failed: {ret}'
    print('[✓] DVPP 通道创建成功')

    # === 依次对四张图片执行 DVPP 解码 + 缩放 ===
    dvpp_results = []  # 保存每张图片的耗时
    for t in test_images:
        jpg_path = t['jpg']
        label = t['label']
        print(f'\n--- 处理 {label}: {jpg_path} ---')

        # 读取 JPEG
        with open(jpg_path, 'rb') as f:
            jpeg_data = f.read()
        jpeg_np = np.frombuffer(jpeg_data, dtype=np.byte)
        jpeg_ptr = acl.util.bytes_to_ptr(jpeg_data)

        # 获取图片信息
        width, height, _, ret = acl.media.dvpp_jpeg_get_image_info(jpeg_ptr, len(jpeg_data))
        assert ret == 0
        print(f'  JPEG 原始尺寸: {width}x{height}')

        # JPEG → Device
        dev_jpeg, ret = acl.media.dvpp_malloc(len(jpeg_data))
        assert ret == 0
        acl.rt.memcpy(dev_jpeg, len(jpeg_data), jpeg_ptr, len(jpeg_data), ACL_MEMCPY_HOST_TO_DEVICE)

        # DVPP JPEG 解码 (JPEG → YUV420SP)
        aw, ah = align_up(width, 128), align_up(height, 16)
        yuv_size = (aw * ah * 3) // 2
        out_desc = acl.media.dvpp_create_pic_desc()
        dev_yuv, ret = acl.media.dvpp_malloc(yuv_size)
        assert ret == 0
        acl.media.dvpp_set_pic_desc_data(out_desc, dev_yuv)
        acl.media.dvpp_set_pic_desc_format(out_desc, PIXEL_FORMAT_YUV_SEMIPLANAR_420)
        acl.media.dvpp_set_pic_desc_width(out_desc, width)
        acl.media.dvpp_set_pic_desc_height(out_desc, height)
        acl.media.dvpp_set_pic_desc_width_stride(out_desc, aw)
        acl.media.dvpp_set_pic_desc_height_stride(out_desc, ah)
        acl.media.dvpp_set_pic_desc_size(out_desc, yuv_size)

        t0 = time.time()
        ret = acl.media.dvpp_jpeg_decode_async(dvpp_desc, dev_jpeg, len(jpeg_data), out_desc, stream)
        assert ret == 0, f'decode failed: {ret}'
        acl.rt.synchronize_stream(stream)
        decode_ms = (time.time() - t0) * 1000
        print(f'  [✓] DVPP JPEG 解码: {decode_ms:.2f} ms (输出 YUV420SP {aw}x{ah})')

        # DVPP VPC 缩放 (640x480 → 224x224)
        dst_w, dst_h = 224, 224
        dst_aw, dst_ah = align_up(dst_w, 16), align_up(dst_h, 2)
        dst_size = (dst_aw * dst_ah * 3) // 2

        src_desc = acl.media.dvpp_create_pic_desc()
        acl.media.dvpp_set_pic_desc_data(src_desc, dev_yuv)
        acl.media.dvpp_set_pic_desc_format(src_desc, PIXEL_FORMAT_YUV_SEMIPLANAR_420)
        acl.media.dvpp_set_pic_desc_width(src_desc, width)
        acl.media.dvpp_set_pic_desc_height(src_desc, height)
        acl.media.dvpp_set_pic_desc_width_stride(src_desc, aw)
        acl.media.dvpp_set_pic_desc_height_stride(src_desc, ah)
        acl.media.dvpp_set_pic_desc_size(src_desc, yuv_size)

        dst_desc = acl.media.dvpp_create_pic_desc()
        dev_out, ret = acl.media.dvpp_malloc(dst_size)
        assert ret == 0
        acl.media.dvpp_set_pic_desc_data(dst_desc, dev_out)
        acl.media.dvpp_set_pic_desc_format(dst_desc, PIXEL_FORMAT_YUV_SEMIPLANAR_420)
        acl.media.dvpp_set_pic_desc_width(dst_desc, dst_w)
        acl.media.dvpp_set_pic_desc_height(dst_desc, dst_h)
        acl.media.dvpp_set_pic_desc_width_stride(dst_desc, dst_aw)
        acl.media.dvpp_set_pic_desc_height_stride(dst_desc, dst_ah)
        acl.media.dvpp_set_pic_desc_size(dst_desc, dst_size)

        resize_cfg = acl.media.dvpp_create_resize_config()
        t0 = time.time()
        ret = acl.media.dvpp_vpc_resize_async(dvpp_desc, src_desc, dst_desc, resize_cfg, stream)
        assert ret == 0, f'resize failed: {ret}'
        acl.rt.synchronize_stream(stream)
        resize_ms = (time.time() - t0) * 1000
        print(f'  [✓] DVPP VPC 缩放: {resize_ms:.2f} ms ({width}x{height} → {dst_w}x{dst_h})')

        dvpp_results.append({'label': label, 'decode_ms': decode_ms, 'resize_ms': resize_ms})

        # 释放本张图片资源
        acl.media.dvpp_free(dev_jpeg); acl.media.dvpp_free(dev_yuv); acl.media.dvpp_free(dev_out)
        acl.media.dvpp_destroy_pic_desc(out_desc); acl.media.dvpp_destroy_pic_desc(src_desc)
        acl.media.dvpp_destroy_pic_desc(dst_desc); acl.media.dvpp_destroy_resize_config(resize_cfg)

    # 释放通道和 ACL 资源
    acl.media.dvpp_destroy_channel(dvpp_desc); acl.media.dvpp_destroy_channel_desc(dvpp_desc)
    acl.rt.destroy_stream(stream); acl.rt.reset_device(0); acl.finalize()

    print('\n' + '=' * 55)
    print('  DVPP 四图处理结果汇总')
    print('=' * 55)
    for r in dvpp_results:
        total = r['decode_ms'] + r['resize_ms']
        print(f"  {r['label']}: 解码 {r['decode_ms']:.2f} ms + 缩放 {r['resize_ms']:.2f} ms = {total:.2f} ms")
    avg_total = sum(r['decode_ms'] + r['resize_ms'] for r in dvpp_results) / len(dvpp_results)
    print(f'  平均总耗时: {avg_total:.2f} ms/张')
    print('[✓] DVPP 演示完成！')

except Exception as e:
    print(f'DVPP 演示失败: {e}')
    print('（请确认 CANN 环境已正确加载：source /usr/local/Ascend/ascend-toolkit/set_env.sh）')
    # 尝试释放 ACL 框架，避免后续单元格 acl.init() 报 100002
    try:
        acl.rt.reset_device(0)
        acl.finalize()
    except Exception:
        pass

### 6.3 对比 OpenCV 软解码与 DVPP 硬件解码

对比 CPU（OpenCV）和 NPU（DVPP）两种图像预处理方式，使用 dog1.jpg、dog2.jpg、cat1.jpg 和 cat2.jpg 四张图片取平均。

**测试程序说明**：对四张测试图片分别用 OpenCV 执行 100 次「JPEG 解码 → BGR→RGB → 缩放到 224×224」的完整预处理流程并计时，取四张图片的平均值与上一节 DVPP 的解码+缩放耗时对比。先做 5 次热身消除冷启动效应。

**预期结果**：OpenCV 预处理约 2~5 ms/次，DVPP 硬件预处理约 0.5~1.5 ms/次，加速比约 3~5 倍。

**为什么有这样的结果**：OpenCV 的 `imdecode` 在 CPU 上做完整的 JPEG 解码（哈夫曼解码+IDCT），`resize` 做双线性插值缩放，`cvtColor` 做色彩矩阵乘法，三步都在 CPU 通用核心上逐像素计算。DVPP 将这三步合并到专用硬件流水线中，解码用硬件 Huffman 解码器，缩放用硬件插值单元，且数据始终在 Device 内存中不回传 Host，因此显著更快。

In [ ]:
# OpenCV 软件预处理计时（对两张图片取平均）
N = 100

# 确保依赖和变量可用
try:
    cv2
except NameError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'opencv-python-headless', '-q'])
    import cv2

# 确保两张测试图片可用
if 'test_images' not in dir() or not test_images:
    os.makedirs('output', exist_ok=True)
    DVPP_W, DVPP_H = 640, 480
    image_samples = [
        {'png': '../../images/dog1.jpg', 'jpg': 'output/dog1.jpg', 'label': 'Dog1'},
        {'png': '../../images/dog2.jpg', 'jpg': 'output/dog2.jpg', 'label': 'Dog2'},
        {'png': '../../images/cat1.jpg', 'jpg': 'output/cat1.jpg', 'label': 'Cat1'},
        {'png': '../../images/cat2.jpg', 'jpg': 'output/cat2.jpg', 'label': 'Cat2'},
    ]
    test_images = []
    for s in image_samples:
        img = cv2.resize(cv2.imread(s['png']), (DVPP_W, DVPP_H))
        cv2.imwrite(s['jpg'], img)
        test_images.append({'jpg': s['jpg'], 'label': s['label'], 'img': img})
    test_jpg = test_images[0]['jpg']
    print(f'重新生成 {len(test_images)} 张测试图片')

# OpenCV: JPEG → BGR → resize → RGB，对两张图片分别计时后取平均
opencv_ms_list = []
for t in test_images:
    jpeg_bytes = open(t['jpg'], 'rb').read()
    for _ in range(5):  # 热身
        bgr = cv2.imdecode(np.frombuffer(jpeg_bytes, np.byte), cv2.IMREAD_COLOR)
        rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
        rgb = cv2.resize(rgb, (224, 224))
    t0 = time.time()
    for _ in range(N):
        bgr = cv2.imdecode(np.frombuffer(jpeg_bytes, np.byte), cv2.IMREAD_COLOR)
        rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
        rgb = cv2.resize(rgb, (224, 224))
    ms = (time.time() - t0) / N * 1000
    opencv_ms_list.append(ms)
    print(f"  OpenCV 预处理 {t['label']}: {ms:.2f} ms/次")

opencv_ms = sum(opencv_ms_list) / len(opencv_ms_list)
print(f'\nOpenCV 预处理平均 (解码+缩放): {opencv_ms:.2f} ms/次 (dog1 + cat2 平均)')

# DVPP 平均耗时
if 'dvpp_results' in dir() and dvpp_results:
    dvpp_total = sum(r['decode_ms'] + r['resize_ms'] for r in dvpp_results) / len(dvpp_results)
    print(f'DVPP 硬件预处理平均 (解码+缩放): {dvpp_total:.2f} ms/次')
    print(f'加速比: {opencv_ms / dvpp_total:.1f}x')
else:
    dvpp_total = 0
    print('DVPP 硬件预处理: 未运行（请先执行 DVPP 演示单元格）')
print('\n结论: DVPP 把 JPEG 解码和图像缩放从 CPU 移到 NPU 专用硬件，显著降低预处理延迟。')

# === 性能对比柱状图 ===
try:
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    labels = ['OpenCV\n(CPU Software)']
    vals = [opencv_ms]
    cols = ['#4C72B0']
    if dvpp_total > 0:
        labels.append('DVPP\n(NPU Hardware)')
        vals.append(dvpp_total)
        cols.append('#DD8452')
    fig, ax = plt.subplots(figsize=(6, 4))
    bars = ax.bar(labels, vals, color=cols, width=0.5, edgecolor='black')
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05, f'{v:.2f} ms', ha='center', fontsize=12)
    ax.set_ylabel('Preprocessing Time (ms)', fontsize=12)
    ax.set_title('OpenCV vs DVPP Preprocessing (avg of 4 images)', fontsize=14)
    if len(vals) > 1:
        ax.annotate(f'{vals[0]/vals[1]:.1f}x Speedup', xy=(1, vals[1]), xytext=(0.5, max(vals)*0.7),
                    fontsize=14, color='red', fontweight='bold',
                    arrowprops=dict(arrowstyle='->', color='red', lw=2))
    plt.tight_layout(); plt.savefig('output/opencv_vs_dvpp.png', dpi=150, bbox_inches='tight'); plt.show()
    print('性能对比图已保存: output/opencv_vs_dvpp.png')
except Exception as e:
    print(f'绘图跳过: {e}')

### 6.4 体验 AIPP 模型转换

创建一个简单模型，分别转换为纯 OM 和 AIPP-OM，对比差异。

**测试程序说明**：① 定义 `MiniYOLO` 模型（两层卷积+ReLU+检测头，输入 640×640），导出为 ONNX；② 写入 AIPP 配置文件 `output/aipp.cfg`，配置静态 AIPP、RGB888_U8 输入格式、归一化参数（`var_reci_chn=1/255`）。

**预期结果**：ONNX 导出约 10~30 KB。AIPP 配置文件写入成功。若 PyTorch 因 TLS 问题无法导入，代码会自动预加载 `libgomp` 修复。

**为什么有这样的结果**：MiniYOLO 的检测头输出 5 通道（cx, cy, w, h, conf），这是 YOLO 检测框的标准表示。AIPP 配置中 `input_format: RGB888_U8` 告诉 ATC 模型的输入是 uint8 类型的 RGB 图像，ATC 会在模型头部插入 AIPP 预处理算子，将 uint8 自动转为 float32 并归一化到 [0, 1]。

In [ ]:
# 修复 torch TLS 加载问题：预加载 libgomp 到全局符号表
try:
    import ctypes, glob
    for p in glob.glob('/opt/**/torch.libs/libgomp*.so*', recursive=True):
        try:
            ctypes.CDLL(p, mode=ctypes.RTLD_GLOBAL)
        except Exception:
            pass
except Exception:
    pass

import torch
try:
    import torch_npu
except ImportError:
    pass

# 确保 onnx 可用
try:
    import onnx
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'onnx', '-q'])

# 创建一个模拟 YOLOv8 检测头的小模型
class MiniYOLO(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = torch.nn.Conv2d(3, 16, 3, padding=1)
        self.conv2 = torch.nn.Conv2d(16, 32, 3, padding=1)
        self.conv3 = torch.nn.Conv2d(32, 5, 1)  # 5 = cx,cy,w,h,conf
    def forward(self, x):
        x = torch.relu(self.conv1(x))
        x = torch.relu(self.conv2(x))
        return self.conv3(x)

model = MiniYOLO().eval()
dummy = torch.randn(1, 3, 640, 640)
onnx_path = 'output/mini_yolo.onnx'
torch.onnx.export(model, dummy, onnx_path, input_names=['images'],
                   output_names=['output'], opset_version=13)
print(f'ONNX 导出: {onnx_path} ({os.path.getsize(onnx_path)//1024} KB)')

# 写入 AIPP 配置
aipp_cfg = """aipp_op {
  aipp_mode: static
  related_input_rank: 0
  input_format: RGB888_U8
  src_image_size_w: 640
  src_image_size_h: 640
  crop: false
  load_start_pos_h: 0
  load_start_pos_w: 0
  csc_switch: false
  min_chn_0: 0.0
  min_chn_1: 0.0
  min_chn_2: 0.0
  var_reci_chn_0: 0.00392157
  var_reci_chn_1: 0.00392157
  var_reci_chn_2: 0.00392157
}
"""
with open('output/aipp.cfg', 'w') as f:
    f.write(aipp_cfg)
print('AIPP 配置已写入: output/aipp.cfg')

In [ ]:
# 确保 onnx_path 可用，若前序单元格未执行则重新导出
if 'onnx_path' not in dir() or not os.path.exists(onnx_path if 'onnx_path' in dir() else ''):
    try:
        import ctypes, glob
        for p in glob.glob('/opt/**/torch.libs/libgomp*.so*', recursive=True):
            try: ctypes.CDLL(p, mode=ctypes.RTLD_GLOBAL)
            except: pass
    except: pass
    import torch
    os.makedirs('output', exist_ok=True)
    class MiniYOLO(torch.nn.Module):
        def __init__(self):
            super().__init__()
            self.conv1 = torch.nn.Conv2d(3, 16, 3, padding=1)
            self.conv2 = torch.nn.Conv2d(16, 32, 3, padding=1)
            self.conv3 = torch.nn.Conv2d(32, 5, 1)
        def forward(self, x):
            x = torch.relu(self.conv1(x))
            x = torch.relu(self.conv2(x))
            return self.conv3(x)
    _m = MiniYOLO().eval()
    onnx_path = 'output/mini_yolo.onnx'
    torch.onnx.export(_m, torch.randn(1, 3, 640, 640), onnx_path,
                      input_names=['images'], output_names=['output'], opset_version=13)
    print(f'重新导出 ONNX: {onnx_path}')

# ATC 转换：纯 OM
print('正在转换为纯 OM（无 AIPP）...')
ret = os.system(
    f'atc --model={onnx_path} --framework=5 --output=output/mini_yolo '
    f'--input_shape="images:1,3,640,640" --soc_version=Ascend910B3 --log=error 2>&1 | tail -3'
)
om_pure = 'output/mini_yolo.om'
if os.path.exists(om_pure):
    print(f'纯 OM: {os.path.getsize(om_pure)//1024} KB')
else:
    print('纯 OM 转换失败（请检查 CANN 环境）')

# ATC 转换：AIPP-OM
print('\n正在转换为 AIPP-OM...')
ret = os.system(
    f'atc --model={onnx_path} --framework=5 --output=output/mini_yolo_aipp '
    f'--input_shape="images:1,3,640,640" --soc_version=Ascend910B3 '
    f'--insert_op_conf=output/aipp.cfg --log=error 2>&1 | tail -3'
)
om_aipp = 'output/mini_yolo_aipp.om'
if os.path.exists(om_aipp):
    print(f'AIPP-OM: {os.path.getsize(om_aipp)//1024} KB')
else:
    print('AIPP-OM 转换失败')

### 6.5 用 AscendCL 加载 OM 推理：纯 OM vs AIPP-OM 对比

演示 AscendCL 的完整推理生命周期，并**对比纯 OM（无 AIPP）与 AIPP-OM 的推理性能差异**。

**测试程序说明**：使用 AscendCL Python API 分别加载纯 OM 和 AIPP-OM，各运行 20 次推理取平均耗时：① `acl.init()` 初始化框架；② 加载两个 OM 模型并获取输入输出尺寸；③ 纯 OM 输入为 float32（4.9 MB），AIPP-OM 输入为 uint8（1.2 MB，降为 1/4）；④ 各运行 20 次推理计时；⑤ 对比推理耗时和输入数据量；⑥ 绘制英文对比柱状图。

**预期结果**：AIPP-OM 输入数据量仅为纯 OM 的 1/4（1.2 MB vs 4.9 MB）。AIPP-OM 推理耗时略高于纯 OM（因 AIPP 算子额外做归一化和转置），但端到端（含 Host→Device 传输）AIPP-OM 更快。

**为什么有这样的结果**：AIPP 在模型头部插入了硬件预处理算子，将 uint8 输入自动转为 float32 并归一化。这使得 Host→Device 传输量降为 1/4，在带宽受限的端侧场景下收益显著。纯 OM 虽然推理本身略快，但需传输 4 倍的 float32 数据。

In [ ]:
# AscendCL OM 推理演示：纯 OM vs AIPP-OM 性能对比
om_pure = 'output/mini_yolo.om'
om_aipp = 'output/mini_yolo_aipp.om'
RUNS = 20  # 每个模型推理次数，取平均

def run_om_inference(om_path, runs=20, use_uint8=False):
    """加载 OM 模型并执行多次推理，返回平均耗时和输出统计"""
    ACL_MEM_MALLOC_NORMAL_ONLY = 2
    ACL_MEMCPY_HOST_TO_DEVICE = 1
    ACL_MEMCPY_DEVICE_TO_HOST = 2

    # 加载模型
    model_id, ret = acl.mdl.load_from_file(om_path)
    assert ret == 0, f'load failed: {ret}'
    model_desc = acl.mdl.create_desc()
    acl.mdl.get_desc(model_desc, model_id)

    input_size = acl.mdl.get_input_size_by_index(model_desc, 0)
    output_size = acl.mdl.get_output_size_by_index(model_desc, 0)

    # 分配设备内存
    input_dev, _ = acl.rt.malloc(input_size, ACL_MEM_MALLOC_NORMAL_ONLY)
    output_dev, _ = acl.rt.malloc(output_size, ACL_MEM_MALLOC_NORMAL_ONLY)

    in_dataset = acl.mdl.create_dataset()
    in_buf = acl.create_data_buffer(input_dev, input_size)
    acl.mdl.add_dataset_buffer(in_dataset, in_buf)

    out_dataset = acl.mdl.create_dataset()
    out_buf = acl.create_data_buffer(output_dev, output_size)
    acl.mdl.add_dataset_buffer(out_dataset, out_buf)

    # 准备输入：AIPP-OM 用 uint8，纯 OM 用 float32
    if use_uint8:
        img_input = np.random.randint(0, 256, (1, 3, 640, 640), dtype=np.uint8)
    else:
        img_input = np.random.randn(1, 3, 640, 640).astype(np.float32)
    acl.rt.memcpy(input_dev, input_size, img_input.ctypes.data,
                  input_size, ACL_MEMCPY_HOST_TO_DEVICE)

    # 热身 3 次
    for _ in range(3):
        acl.mdl.execute(model_id, in_dataset, out_dataset)

    # 正式计时
    t0 = time.time()
    for _ in range(runs):
        ret = acl.mdl.execute(model_id, in_dataset, out_dataset)
        assert ret == 0, f'execute failed: {ret}'
    infer_ms = (time.time() - t0) / runs * 1000

    # 拷贝输出回 Host
    out_np = np.zeros(output_size, dtype=np.uint8)
    acl.rt.memcpy(out_np.ctypes.data, output_size, output_dev,
                  output_size, ACL_MEMCPY_DEVICE_TO_HOST)
    out_data = out_np.view(np.float32).reshape(1, 5, 640, 640)

    # 释放资源
    acl.destroy_data_buffer(in_buf); acl.destroy_data_buffer(out_buf)
    acl.mdl.destroy_dataset(in_dataset); acl.mdl.destroy_dataset(out_dataset)
    acl.rt.free(input_dev); acl.rt.free(output_dev)
    acl.mdl.destroy_desc(model_desc); acl.mdl.unload(model_id)

    return {
        'infer_ms': infer_ms,
        'input_size': input_size,
        'output_size': output_size,
        'out_mean': float(out_data.mean()),
        'out_std': float(out_data.std()),
    }

comparison_results = {}
if os.path.exists(om_pure) or os.path.exists(om_aipp):
    try:
        import acl

        # 初始化（容忍已初始化的情况，错误码 100002 = ACL_ERROR_REPEAT_INITIALIZE）
        ret = acl.init()
        assert ret in (0, 100002), f'acl.init failed: {ret}'
        acl.rt.set_device(0)
        context, _ = acl.rt.create_context(0)
        stream, _ = acl.rt.create_stream()

        # --- 纯 OM 推理 ---
        if os.path.exists(om_pure):
            print(f'>>> Pure OM inference ({RUNS} runs)...')
            r_pure = run_om_inference(om_pure, RUNS, use_uint8=False)
            comparison_results['pure'] = r_pure
            print(f'    Input:  {r_pure["input_size"]} bytes ({r_pure["input_size"]/1024/1024:.2f} MB, float32)')
            print(f'    Output: {r_pure["output_size"]} bytes')
            print(f'    Avg inference: {r_pure["infer_ms"]:.2f} ms')
            print(f'    Output stats: mean={r_pure["out_mean"]:.4f}, std={r_pure["out_std"]:.4f}')
        else:
            print('Pure OM not found, skipping.')

        # --- AIPP-OM 推理 ---
        if os.path.exists(om_aipp):
            print(f'\n>>> AIPP-OM inference ({RUNS} runs)...')
            r_aipp = run_om_inference(om_aipp, RUNS, use_uint8=True)
            comparison_results['aipp'] = r_aipp
            print(f'    Input:  {r_aipp["input_size"]} bytes ({r_aipp["input_size"]/1024/1024:.2f} MB, uint8)')
            print(f'    Output: {r_aipp["output_size"]} bytes')
            print(f'    Avg inference: {r_aipp["infer_ms"]:.2f} ms')
            print(f'    Output stats: mean={r_aipp["out_mean"]:.4f}, std={r_aipp["out_std"]:.4f}')
        else:
            print('AIPP-OM not found, skipping.')

        # 释放 ACL 资源
        acl.rt.destroy_stream(stream); acl.rt.destroy_context(context)
        acl.rt.reset_device(0); acl.finalize()
        print('\n[✓] AscendCL inference completed, resources released.')

    except Exception as e:
        print(f'Inference failed: {e}')
        try:
            acl.rt.reset_device(0)
            acl.finalize()
        except Exception:
            pass
else:
    print(f'OM models not found: {om_pure}, {om_aipp} (please run ATC conversion cells first)')

# === 纯 OM vs AIPP-OM 对比柱状图（全英文） ===
if len(comparison_results) >= 1:
    try:
        import matplotlib
        matplotlib.use('Agg')
        import matplotlib.pyplot as plt

        fig, axes = plt.subplots(1, 2, figsize=(14, 5))

        # 子图1：推理耗时对比
        ax1 = axes[0]
        labels_t = []
        vals_t = []
        cols_t = []
        if 'pure' in comparison_results:
            labels_t.append('Pure OM\n(float32 input)')
            vals_t.append(comparison_results['pure']['infer_ms'])
            cols_t.append('#4C72B0')
        if 'aipp' in comparison_results:
            labels_t.append('AIPP-OM\n(uint8 input)')
            vals_t.append(comparison_results['aipp']['infer_ms'])
            cols_t.append('#DD8452')
        bars = ax1.bar(labels_t, vals_t, color=cols_t, width=0.5, edgecolor='black')
        for bar, v in zip(bars, vals_t):
            ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                     f'{v:.2f} ms', ha='center', fontsize=12)
        ax1.set_ylabel('Inference Time (ms)', fontsize=12)
        ax1.set_title('Inference Time: Pure OM vs AIPP-OM', fontsize=13)

        # 子图2：输入数据量对比
        ax2 = axes[1]
        labels_s = []
        vals_s = []
        cols_s = []
        if 'pure' in comparison_results:
            labels_s.append('Pure OM\n(float32)')
            vals_s.append(comparison_results['pure']['input_size'] / 1024 / 1024)
            cols_s.append('#4C72B0')
        if 'aipp' in comparison_results:
            labels_s.append('AIPP-OM\n(uint8)')
            vals_s.append(comparison_results['aipp']['input_size'] / 1024 / 1024)
            cols_s.append('#DD8452')
        bars2 = ax2.bar(labels_s, vals_s, color=cols_s, width=0.5, edgecolor='black')
        for bar, v in zip(bars2, vals_s):
            ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
                     f'{v:.2f} MB', ha='center', fontsize=12)
        ax2.set_ylabel('Input Data Size (MB)', fontsize=12)
        ax2.set_title('Input Data Size: Pure OM vs AIPP-OM', fontsize=13)
        if len(vals_s) == 2 and vals_s[1] > 0:
            ax2.annotate(f'{vals_s[0]/vals_s[1]:.1f}x reduction',
                        xy=(1, vals_s[1]), xytext=(0.5, max(vals_s)*0.7),
                        fontsize=13, color='red', fontweight='bold',
                        arrowprops=dict(arrowstyle='->', color='red', lw=2))

        plt.suptitle('Pure OM vs AIPP-OM Comparison', fontsize=14, fontweight='bold')
        plt.tight_layout()
        plt.savefig('output/om_vs_aipp_comparison.png', dpi=150, bbox_inches='tight')
        plt.show()
        print('Comparison chart saved: output/om_vs_aipp_comparison.png')
    except Exception as e:
        print(f'Plot skipped: {e}')

### 6.6 案例二：视频流目标检测（dog.mp4）

在实际视觉系统中，输入通常是视频流而非单张图片。下方读取 `images/dog.mp4` 真实视频的**所有帧**并逐帧处理。视频原始分辨率和帧率由 OpenCV 自动读取，处理时保持原始分辨率不变，检测结果帧合成为结果视频。

**测试程序说明**：① 使用 OpenCV `VideoCapture` 读取 `images/dog.mp4` 真实视频，获取帧数、分辨率、帧率等信息；② 读取**全部帧**并逐帧做模拟目标检测（在画面中标注检测框）；③ 将检测结果帧合成为结果视频并显示 **9 个关键帧**展示检测过程。

**预期结果**：成功读取真实视频并显示视频信息（帧数、分辨率、帧率）。结果视频在每帧标注检测框，显示 9 个关键帧展示检测过程。

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# === 案例二：读取真实测试视频 dog.mp4 的所有帧 ===
video_path = 'images/dog.mp4'
cap = cv2.VideoCapture(video_path)
if not cap.isOpened():
    raise FileNotFoundError(f'无法打开视频: {video_path}')
fps = cap.get(cv2.CAP_PROP_FPS)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
orig_w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
orig_h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
print(f'案例二 - 视频信息: {video_path} ({total_frames} frames, {orig_w}x{orig_h}, {fps:.1f} fps)')

# 读取所有帧
frames = []
while True:
    ret, frame = cap.read()
    if not ret:
        break
    frames.append(frame)
cap.release()
print(f'读取全部 {len(frames)} 帧')

# === 逐帧模拟检测 ===
result_video_path = 'output/demo_video_result.mp4'
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
h, w = frames[0].shape[:2]
vw2 = cv2.VideoWriter(result_video_path, fourcc, fps if fps > 0 else 15, (w, h))
key_frames = []
# 9 个关键帧索引：均匀分布在全部帧中
key_indices = [int(len(frames) * k / 9) for k in range(9)]
for i, frame in enumerate(frames):
    result = frame.copy()
    # 模拟检测框（在实际系统中由 YOLO 模型输出）
    cx, cy = w // 2, h // 2
    bw, bh = int(w * 0.35), int(h * 0.45)
    cv2.rectangle(result, (cx - bw // 2, cy - bh // 2), (cx + bw // 2, cy + bh // 2), (0, 255, 0), 2)
    cv2.rectangle(result, (cx - bw // 2, cy - bh // 2 - 25), (cx + bw // 2 + 20, cy - bh // 2), (0, 255, 0), -1)
    cv2.putText(result, f'Dog 0.90 Frame {i+1}/{len(frames)}',
                (cx - bw // 2 + 3, cy - bh // 2 - 8), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 1)
    vw2.write(result)
    if i in key_indices:
        key_frames.append(result.copy())
vw2.release()
print(f'检测结果视频: {result_video_path} ({os.path.getsize(result_video_path)//1024} KB)')

# === 显示 9 个关键帧（3x3 网格） ===
fig, axes = plt.subplots(3, 3, figsize=(18, 12))
for ax, kf, idx in zip(axes.flat, key_frames, key_indices):
    ax.imshow(cv2.cvtColor(kf, cv2.COLOR_BGR2RGB))
    ax.set_title(f'Frame {idx+1}/{len(frames)}'); ax.axis('off')
plt.suptitle(f'Case 2: Video Stream Detection ({len(frames)} frames, 9 key frames from dog.mp4)', fontsize=14)
plt.tight_layout(); plt.savefig('output/video_detection_demo.png', dpi=150, bbox_inches='tight'); plt.show()
print('视频检测可视化已保存: output/video_detection_demo.png')

### 6.7 案例二扩展：DeepSort 多目标追踪

在目标检测基础上，使用 **DeepSort** 算法实现多目标连续追踪。DeepSort 结合卡尔曼滤波（运动预测）和匈牙利匹配（数据关联），为每个检测目标分配唯一的 Track ID，实现跨帧连续追踪。

**测试程序说明**：① 实现简化版 DeepSort 追踪器（卡尔曼滤波 + IoU 匹配）；② 对 `dog.mp4` 全部帧的模拟检测结果进行追踪；③ 为每个目标分配 Track ID 并绘制追踪框和轨迹；④ 生成追踪结果视频并显示 9 个关键帧。

**预期结果**：追踪器为检测目标分配稳定的 Track ID，追踪结果视频展示带 ID 的追踪框和运动轨迹。9 个关键帧展示追踪过程。

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from collections import defaultdict

# ============================================================
# 简化版 DeepSort 追踪器：卡尔曼滤波 + IoU 匹配
# ============================================================
class KalmanBoxTracker:
    """单目标卡尔曼追踪器（简化版，仅追踪 bbox 中心点和宽高）"""
    _count = 0

    def __init__(self, bbox):
        # bbox = [x1, y1, x2, y2]
        self.id = KalmanBoxTracker._count
        KalmanBoxTracker._count += 1
        self.time_since_update = 0
        self.hits = 1
        # 状态：[cx, cy, w, h, vx, vy]
        self.cx = (bbox[0] + bbox[2]) / 2.0
        self.cy = (bbox[1] + bbox[3]) / 2.0
        self.w = bbox[2] - bbox[0]
        self.h = bbox[3] - bbox[1]
        self.vx = 0.0
        self.vy = 0.0
        self.history = [(self.cx, self.cy)]  # 轨迹历史

    def predict(self):
        """卡尔曼预测步骤：匀速运动模型"""
        self.cx += self.vx
        self.cy += self.vy
        self.time_since_update += 1
        return self.get_bbox()

    def update(self, bbox):
        """卡尔曼更新步骤：用新观测修正状态"""
        new_cx = (bbox[0] + bbox[2]) / 2.0
        new_cy = (bbox[1] + bbox[3]) / 2.0
        new_w = bbox[2] - bbox[0]
        new_h = bbox[3] - bbox[1]
        # 简单卡尔曼增益：融合预测和观测
        alpha = 0.5  # 观测权重
        old_cx, old_cy = self.cx, self.cy
        self.cx = alpha * new_cx + (1 - alpha) * self.cx
        self.cy = alpha * new_cy + (1 - alpha) * self.cy
        self.w = alpha * new_w + (1 - alpha) * self.w
        self.h = alpha * new_h + (1 - alpha) * self.h
        self.vx = self.cx - old_cx
        self.vy = self.cy - old_cy
        self.time_since_update = 0
        self.hits += 1
        self.history.append((self.cx, self.cy))
        if len(self.history) > 30:  # 最多保留 30 帧轨迹
            self.history.pop(0)

    def get_bbox(self):
        """返回当前预测的 bbox [x1, y1, x2, y2]"""
        x1 = self.cx - self.w / 2.0
        y1 = self.cy - self.h / 2.0
        x2 = self.cx + self.w / 2.0
        y2 = self.cy + self.h / 2.0
        return [x1, y1, x2, y2]


def iou(bbox1, bbox2):
    """计算两个 bbox 的 IoU"""
    x1 = max(bbox1[0], bbox2[0])
    y1 = max(bbox1[1], bbox2[1])
    x2 = min(bbox1[2], bbox2[2])
    y2 = min(bbox1[3], bbox2[3])
    inter = max(0, x2 - x1) * max(0, y2 - y1)
    area1 = (bbox1[2] - bbox1[0]) * (bbox1[3] - bbox1[1])
    area2 = (bbox2[2] - bbox2[0]) * (bbox2[3] - bbox2[1])
    union = area1 + area2 - inter
    return inter / union if union > 0 else 0


class DeepSortTracker:
    """简化版 DeepSort 追踪器"""
    def __init__(self, max_age=5, iou_threshold=0.3):
        self.trackers = []
        self.max_age = max_age
        self.iou_threshold = iou_threshold

    def update(self, detections):
        """
        detections: list of [x1, y1, x2, y2, score, label]
        returns: list of [x1, y1, x2, y2, track_id, label]
        """
        # 预测所有追踪器
        predicted = [t.predict() for t in self.trackers]

        # 匹配：贪心 IoU 匹配
        matched = []
        unmatched_det = list(range(len(detections)))
        unmatched_trk = list(range(len(self.trackers)))

        # 计算所有 IoU 并按降序匹配
        iou_pairs = []
        for d_idx in range(len(detections)):
            for t_idx in range(len(self.trackers)):
                score = iou(detections[d_idx][:4], predicted[t_idx])
                if score > self.iou_threshold:
                    iou_pairs.append((score, d_idx, t_idx))
        iou_pairs.sort(reverse=True)

        for score, d_idx, t_idx in iou_pairs:
            if d_idx in unmatched_det and t_idx in unmatched_trk:
                self.trackers[t_idx].update(detections[d_idx][:4])
                matched.append((d_idx, t_idx))
                unmatched_det.remove(d_idx)
                unmatched_trk.remove(t_idx)

        # 未匹配的检测 -> 新建追踪器
        for d_idx in unmatched_det:
            self.trackers.append(KalmanBoxTracker(detections[d_idx][:4]))

        # 删除超时的追踪器
        self.trackers = [t for t in self.trackers if t.time_since_update <= self.max_age]

        # 返回活跃追踪结果
        results = []
        for t in self.trackers:
            if t.time_since_update == 0 and t.hits >= 1:
                bbox = t.get_bbox()
                results.append([int(bbox[0]), int(bbox[1]), int(bbox[2]), int(bbox[3]), t.id])
        return results


# ============================================================
# 对 dog.mp4 全部帧执行 DeepSort 追踪
# ============================================================
KalmanBoxTracker._count = 0  # 重置 ID 计数器
tracker = DeepSortTracker(max_age=5, iou_threshold=0.3)

# 为演示多目标追踪，模拟 2 个目标：一个中心目标 + 一个移动目标
track_video_path = 'output/deepsort_tracking_result.mp4'
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
h, w = frames[0].shape[:2]
vw_track = cv2.VideoWriter(track_video_path, fourcc, fps if fps > 0 else 15, (w, h))

track_key_frames = []
track_key_indices = [int(len(frames) * k / 9) for k in range(9)]
track_colors = [(0, 255, 0), (255, 0, 0), (0, 0, 255), (255, 255, 0), (255, 0, 255)]

print('DeepSort 追踪开始...')
for i, frame in enumerate(frames):
    result = frame.copy()

    # 模拟 2 个检测目标
    # 目标1：中心目标（基本静止）
    cx1, cy1 = w // 2, h // 2
    bw1, bh1 = int(w * 0.3), int(h * 0.4)
    det1 = [cx1 - bw1 // 2, cy1 - bh1 // 2, cx1 + bw1 // 2, cy1 + bh1 // 2, 0.92, 'Dog']

    # 目标2：移动目标（沿对角线移动）
    progress = i / max(len(frames) - 1, 1)
    cx2 = int(w * 0.2 + (w * 0.6) * progress)
    cy2 = int(h * 0.2 + (h * 0.6) * progress)
    bw2, bh2 = int(w * 0.2), int(h * 0.3)
    det2 = [cx2 - bw2 // 2, cy2 - bh2 // 2, cx2 + bw2 // 2, cy2 + bh2 // 2, 0.85, 'Dog']

    detections = [det1, det2]
    tracks = tracker.update(detections)

    # 绘制追踪结果
    for trk in tracks:
        x1, y1, x2, y2, tid = trk
        color = track_colors[tid % len(track_colors)]
        # 追踪框
        cv2.rectangle(result, (x1, y1), (x2, y2), color, 2)
        # Track ID 标签
        label_text = f'ID:{tid} Dog'
        cv2.rectangle(result, (x1, y1 - 22), (x1 + 80, y1), color, -1)
        cv2.putText(result, label_text, (x1 + 3, y1 - 6),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)

        # 绘制轨迹线
        for t in tracker.trackers:
            if t.id == tid and len(t.history) > 1:
                for j in range(1, len(t.history)):
                    pt1 = (int(t.history[j-1][0]), int(t.history[j-1][1]))
                    pt2 = (int(t.history[j][0]), int(t.history[j][1]))
                    cv2.line(result, pt1, pt2, color, 1)

    # 帧编号
    cv2.putText(result, f'Frame {i+1}/{len(frames)} Tracks:{len(tracks)}',
                (10, 25), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255), 2)

    vw_track.write(result)
    if i in track_key_indices:
        track_key_frames.append(result.copy())

vw_track.release()
print(f'DeepSort 追踪完成！结果视频: {track_video_path} ({os.path.getsize(track_video_path)//1024} KB)')
print(f'总帧数: {len(frames)}, 最终追踪目标数: {len(tracker.trackers)}')

# === 显示 9 个追踪关键帧（3x3 网格） ===
fig, axes = plt.subplots(3, 3, figsize=(18, 12))
for ax, kf, idx in zip(axes.flat, track_key_frames, track_key_indices):
    ax.imshow(cv2.cvtColor(kf, cv2.COLOR_BGR2RGB))
    ax.set_title(f'Tracking Frame {idx+1}/{len(frames)}'); ax.axis('off')
plt.suptitle('Case 2: DeepSort Multi-Object Tracking (dog.mp4, 9 key frames)', fontsize=14)
plt.tight_layout(); plt.savefig('output/deepsort_tracking_demo.png', dpi=150, bbox_inches='tight'); plt.show()
print('DeepSort 追踪可视化已保存: output/deepsort_tracking_demo.png')

## 7. 实验代码文件说明

本实验的全部代码均集成在 **`02_dvpp_aipp_yolo_deepsort.ipynb`** Notebook 文件中，文件夹内仅包含此一个 Notebook 文件（及 `images/` 测试资源目录）。

<table style="text-align: left; margin-left: 0;">
<tr><th style="text-align: left;">文件</th><th style="text-align: left;">功能说明</th></tr>
<tr><td style="text-align: left;">`02_dvpp_aipp_yolo_deepsort.ipynb`</td><td style="text-align: left;">本 Notebook，包含全部实验代码：环境检查、DVPP 硬件解码、AIPP 模型转换、OM 推理对比、视频检测与 DeepSort 追踪</td></tr>
</table>

上表列出了实验文件夹中的唯一代码文件。`02_dvpp_aipp_yolo_deepsort.ipynb` 是一个自包含的 Jupyter Notebook，所有实验步骤均以单元格形式组织：

- **6.1 节**：检查 gitcode CANNLab 云平台环境（NPU 910B3、ATC、AscendCL）
- **6.2 节**（案例一）：对 dog1.jpg、dog2.jpg、cat1.jpg、cat2.jpg 四张图片执行 DVPP 硬件解码与缩放
- **6.3 节**：对比 OpenCV 软解码与 DVPP 硬件解码性能
- **6.4 节**：体验 AIPP 模型转换（纯 OM vs AIPP-OM）
- **6.5 节**：用 AscendCL 加载 OM 推理并对比纯 OM vs AIPP-OM 性能
- **6.6 节**（案例二）：对 dog.mp4 视频逐帧目标检测，显示 9 个关键帧
- **6.7 节**（案例二扩展）：DeepSort 多目标追踪，生成完整追踪结果视频

### 在 gitcode CANNLab 云平台上的运行流程

```bash
# 1. 在 gitcode CANNLab 平台创建 ASCEND 910B3 实例
#    硬件配置: ASCEND, 1*NPU 910B3, 16vCPUs, 32GiB

# 2. 加载 CANN 环境
source /usr/local/Ascend/ascend-toolkit/set_env.sh

# 3. 启动 Jupyter Notebook 运行实验
jupyter notebook 02_dvpp_aipp_yolo_deepsort.ipynb
```

## 8. 常见问题与故障排查

<table style="text-align: left; margin-left: 0;">
<tr><th style="text-align: left;">现象</th><th style="text-align: left;">可能原因</th><th style="text-align: left;">解决方法</th></tr>
<tr><td style="text-align: left;">`atc: command not found`</td><td style="text-align: left;">CANN 环境未加载</td><td style="text-align: left;">`source /usr/local/Ascend/ascend-toolkit/set_env.sh`</td></tr>
<tr><td style="text-align: left;">DVPP 解码失败</td><td style="text-align: left;">输入图片格式不支持</td><td style="text-align: left;">转为 JPEG/YUV 格式后重试</td></tr>
<tr><td style="text-align: left;">AIPP 配置不生效</td><td style="text-align: left;">`aipp.cfg` 参数错误</td><td style="text-align: left;">检查 `--insert_op_conf` 参数与归一化配置</td></tr>
<tr><td style="text-align: left;">NPU 利用率低</td><td style="text-align: left;">预处理仍在 CPU</td><td style="text-align: left;">确认 AIPP 已固化进 OM 模型</td></tr>
<tr><td style="text-align: left;">DVPP 对齐错误</td><td style="text-align: left;">宽高未按 128/16 对齐</td><td style="text-align: left;">使用 `align_up()` 函数对齐</td></tr>
</table>

上表列出了端侧部署中最常见的五类问题。**`atc: command not found`** 是最常见的环境问题，原因是 CANN 的环境变量未加载，解决方法是执行 `source set_env.sh`。**DVPP 解码失败**通常是因为输入图片格式不在 DVPP 支持范围内（DVPP 只支持 JPEG 和特定 YUV 格式，不支持 PNG/BMP）。**AIPP 配置不生效**的典型原因是 `--insert_op_conf` 参数路径错误或 `aipp.cfg` 中归一化参数写反。**NPU 利用率低**说明预处理仍在 CPU 上做，需确认 OM 模型是用 `--insert_op_conf` 编译的 AIPP-OM 而非纯 OM。**DVPP 对齐错误**是初学者最常踩的坑，DVPP 硬件要求宽度按 128 字节对齐、高度按 16 字节对齐，使用 `align_up(size, align) = (size + align - 1) // align * align` 函数处理。

---

## 小结

本实验在 **gitcode CANNLab 云平台**（ASCEND 1*NPU 910B3, 16vCPUs, 32GiB）上完成了 YOLOv8 目标检测与 DeepSort 多目标追踪，核心实践了：

1. **DVPP 硬件解码**：JPEG → YUV420SP → VPC缩放，全部在 NPU 专用硬件完成
2. **AIPP 预处理卸载**：归一化 /255 和 HWC→CHW 从 CPU 移到 NPU 硬件，输入降为 uint8
3. **四图检测**：对 dog1.jpg、dog2.jpg、cat1.jpg、cat2.jpg 四张图片完成 DVPP 检测
4. **视频检测**：对 dog.mp4 全部帧逐帧检测，展示 9 个关键帧
5. **DeepSort 追踪**：卡尔曼滤波 + IoU 匹配，实现多目标连续追踪并生成追踪结果视频
6. **完整部署链路**：.pt → .onnx → .om（含AIPP）→ AscendCL 推理

> 通过 DVPP 与 AIPP 的协同，端到端推理耗时显著降低，输入数据传输量降为原来的 1/4；结合 DeepSort 实现检测到追踪的完整视觉管线。

---

## 课后练习

**第1题**（单选题）本实验使用的目标硬件和平台是？

- A. 香橙派开发板（昇腾 310B4 NPU）
- B. gitcode CANNLab 云平台（昇腾 910B3 NPU）
- C. NVIDIA GPU
- D. CPU


In [ ]:
q1 = ''  # 填入你的选项，如 'B'
print(f'第{1}题答案已记录：{q1}' if q1 else '⚠️ 请填入答案并运行本单元格')

**第2题**（单选题）DVPP JPEG 解码输出的图像格式是？

- A. RGB888
- B. BGR888
- C. YUV420SP（NV12）
- D. PNG


In [ ]:
q2 = ''  # 填入你的选项，如 'C'
print(f'第{2}题答案已记录：{q2}' if q2 else '⚠️ 请填入答案并运行本单元格')

**第3题**（单选题）DVPP 对输入图片宽度的要求是按多少字节对齐？

- A. 16
- B. 32
- C. 64
- D. 128


In [ ]:
q3 = ''  # 填入你的选项，如 'D'
print(f'第{3}题答案已记录：{q3}' if q3 else '⚠️ 请填入答案并运行本单元格')

**第4题**（单选题）AIPP 配置中 `csc_switch: false` 的原因是？

- A. 不需要色彩转换
- B. BGR→RGB 已由 Python 完成，AIPP 不重复做
- C. CSC 功能不可用
- D. 开启 CSC 会降低精度


In [ ]:
q4 = ''  # 填入你的选项，如 'B'
print(f'第{4}题答案已记录：{q4}' if q4 else '⚠️ 请填入答案并运行本单元格')

**第5题**（单选题）实验对比的三条预处理路径中，哪个使用 NPU 硬件解码？

- A. Path A: OpenCV 公平版
- B. Path B: OpenCV 最优版
- C. Path C: DVPP
- D. 三条路径都用硬件


In [ ]:
q5 = ''  # 填入你的选项，如 'C'
print(f'第{5}题答案已记录：{q5}' if q5 else '⚠️ 请填入答案并运行本单元格')

**第6题**（单选题）启用 AIPP 后，输入数据从 float32 变为 uint8，带来的主要好处是？

- A. 提高计算精度
- B. 增加模型大小
- C. Host→Device 带宽节省为原来的 1/4
- D. 不需要模型转换


In [ ]:
q6 = ''  # 填入你的选项，如 'C'
print(f'第{6}题答案已记录：{q6}' if q6 else '⚠️ 请填入答案并运行本单元格')

**第7题**（单选题）在本实验的 gitcode CANNLab 云平台上，ATC 转换时 `--soc_version` 应设为？

- A. Ascend910B3
- B. Ascend310B4
- C. Ascend910A
- D. GPU


In [ ]:
q7 = ''  # 填入你的选项，如 'A'
print(f'第{7}题答案已记录：{q7}' if q7 else '⚠️ 请填入答案并运行本单元格')

**第8题**（单选题）DVPP VPC 缩放使用的 AscendCL API 是？

- A. acl.media.dvpp_jpeg_decode_async
- B. acl.media.dvpp_vpc_resize_async
- C. acl.mdl.execute
- D. acl.rt.malloc


In [ ]:
q8 = ''  # 填入你的选项，如 'B'
print(f'第{8}题答案已记录：{q8}' if q8 else '⚠️ 请填入答案并运行本单元格')

**全部作答完成后，运行下方代码查看批改结果：**


In [ ]:
import sys
from pathlib import Path

for candidate in (
    Path.cwd() / 'answer',
    Path.cwd() / '06_vision_dev' / 'answer',
):
    if candidate.exists():
        sys.path.insert(0, str(candidate.resolve()))
        break
else:
    raise FileNotFoundError('Cannot find answer directory')
from grade_06 import grade
grade(globals())

## 参考资料

- [实验完整代码](./exp6_ascend_orangepi_local/exp6.1_test_code/)
- [实验手册](./exp6_ascend_orangepi_local/实验6.1_开发板环境基于DVPP与AIPP的YOLO目标检测实验手册.docx)
- [昇腾 DVPP 文档](https://www.hiascend.com/document)
- [昇腾 AIPP 配置指南](https://www.hiascend.com/document/detail/zh/CANNCommunityEdition)
- [YOLOv8 官方仓库](https://github.com/ultralytics/ultralytics)